In [87]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import pandas as pd
import numpy as np

In [29]:
import sys
sys.path.append('../src')

In [94]:
from transformers.transformers_l import DropColumnsTransformer, CityBasedImputer, CityMapTransformer, RollingAverageTransformer
# from transformers.imputation_by_city import CityBasedImputer


In [109]:
X = pd.read_csv('../src/data/raw/dengue_features_train.csv')
y = pd.read_csv('../src/data/raw/dengue_labels_train.csv')
y = y[['total_cases']]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# ndvi_ne,ndvi_nw,ndvi_se,ndvi_sw,precipitation_amt_mm,reanalysis_air_temp_k,reanalysis_avg_temp_k,reanalysis_dew_point_temp_k,reanalysis_max_air_temp_k,
# reanalysis_min_air_temp_k,reanalysis_precip_amt_kg_per_m2,reanalysis_relative_humidity_percent,reanalysis_sat_precip_amt_mm,reanalysis_specific_humidity_g_per_kg,reanalysis_tdtr_k,station_avg_temp_c,
# station_diur_temp_rng_c,station_max_temp_c,station_min_temp_c,station_precip_mm

#create a list of k columns reanalysis_air_temp_k,reanalysis_avg_temp_k,reanalysis_dew_point_temp_k,reanalysis_max_air_temp_k,reanalysis_min_air_temp_k
moving_features = [
    'ndvi_ne',
    'ndvi_sw',
    'precipitation_amt_mm',
    'reanalysis_avg_temp_k',
    'reanalysis_max_air_temp_k',
    'reanalysis_min_air_temp_k',
    # 'reanalysis_precip_amt_kg_per_m2',
    'reanalysis_relative_humidity_percent',
    # 'reanalysis_specific_humidity_g_per_kg',
    'station_avg_temp_c',
    'station_diur_temp_rng_c',
    'station_max_temp_c',
    # 'station_min_temp_c',
    'station_precip_mm'
]

second_drop_columns = [
    'reanalysis_precip_amt_kg_per_m2',
    'station_min_temp_c',
 ]

# Define the pipeline with optimized parameters
pipeline = Pipeline(steps=[
    ('drop_columns', DropColumnsTransformer(columns_to_drop=[
                     'week_start_date', 
                     'reanalysis_sat_prescip_amt_mm',
                     'reanalysis_dew_point_temp_k',
                     'ndvi_nw',
                     'ndvi_se',
                     'reanalysis_air_temp_k',
                     'reanalysis_tdtr_k'])),
    ('city_encoder', CityMapTransformer()),
    ('imputer', CityBasedImputer(city_column='city')),
    # ("city_sel", CitySelector(city=None)),
    # # 3) rolling-mean imputer per city
    # ("rolling_imp", CityWiseRollingMeanImputer(window_size=3, min_periods=1)),
    ('rolling_avg', RollingAverageTransformer(
        columns=moving_features, 
        window=3)),  # Changed from 3 to 2 based on best params
    ('drop_rolling_columns', DropColumnsTransformer(columns_to_drop=moving_features)),  # Keep this to drop original features
    ('drop_extra_columns', DropColumnsTransformer(columns_to_drop=second_drop_columns)),
    ('city_onehot', ColumnTransformer([('onehot', OneHotEncoder(sparse_output=False, drop='first'), ['city'])], remainder='passthrough')), 
    ("scaler", StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=100,  # Changed from 100 to 50
        max_depth=None,   # Already None by default
        min_samples_leaf=1,  # Already 1 by default
        random_state=0))
])


# Fit the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Calculate the mean absolute error
mean_absolute_error(y_test, y_pred)


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


12.47722602739726

In [106]:
X_test_competition  = pd.read_csv('../src/data/raw/dengue_features_test.csv')

pipeline.fit(X, y)
predictions = pipeline.predict(X_test_competition)

# round the predictions to the nearest integer
predictions = np.round(predictions).astype(int)

# Create a DataFrame for the predictions as city,year,weekofyear,total_cases
predictions_df = pd.DataFrame({
    'city': X_test_competition['city'],
    'year': X_test_competition['year'],
    'weekofyear': X_test_competition['weekofyear'],
    'total_cases': predictions
})

# Show predictions
print(predictions_df.head())

# Save the predictions to a CSV file
predictions_df.to_csv('../src/data/predictions/baseline_prediction.csv', index=False)


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


  city  year  weekofyear  total_cases
0   sj  2008          18            5
1   sj  2008          19            5
2   sj  2008          20            6
3   sj  2008          21            6
4   sj  2008          22            7


In [60]:
from sklearn.model_selection import GridSearchCV

moving_features = [
    'ndvi_ne',
    'ndvi_sw',
    'precipitation_amt_mm',
    'reanalysis_avg_temp_k',
    'reanalysis_max_air_temp_k',
    'reanalysis_min_air_temp_k',
    # 'reanalysis_precip_amt_kg_per_m2',
    'reanalysis_relative_humidity_percent',
    # 'reanalysis_specific_humidity_g_per_kg',
    'station_avg_temp_c',
    'station_diur_temp_rng_c',
    'station_max_temp_c',
    # 'station_min_temp_c',
    'station_precip_mm'
]

second_drop_columns = [
    'reanalysis_precip_amt_kg_per_m2',
    'station_min_temp_c',
 ]

# Define the pipeline without the final estimator
pipeline_without_model = Pipeline(steps=[
        ('drop_columns', DropColumnsTransformer(columns_to_drop=[
                     'week_start_date', 
                     'reanalysis_sat_prescip_amt_mm',
                     'reanalysis_dew_point_temp_k',
                     'ndvi_nw',
                     'ndvi_se',
                     'reanalysis_air_temp_k',
                     'reanalysis_tdtr_k'])),
    ('city_encoder', CityMapTransformer()),
    ('imputer', CityBasedImputer(city_column='city')),
    ('rolling_avg', RollingAverageTransformer(
        columns=moving_features, 
        window=3)),
    ('drop_rolling_columns', DropColumnsTransformer(columns_to_drop=moving_features)),
    ('drop_extra_columns', DropColumnsTransformer(columns_to_drop=second_drop_columns)),
    ('city_onehot', ColumnTransformer([('onehot', OneHotEncoder(sparse_output=False, drop='first'), ['city'])], remainder='passthrough'))
])

# Define parameter grid for RandomForestRegressor
param_grid = {
    'n_estimators': [50, 100, 200, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 3, 5]
}

# Create the GridSearchCV object
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=0),
    param_grid=param_grid,
    cv=5,  # 5-fold cross validation
    scoring='neg_mean_absolute_error',
    n_jobs=-1,  # Use all available cores
    verbose=1
)

# Prepare the data with the pipeline
X_train_transformed = pipeline_without_model.fit_transform(X_train)
X_test_transformed = pipeline_without_model.transform(X_test)

# Fit GridSearchCV
grid_search.fit(X_train_transformed, y_train)

# Print results
print(f"Best n_estimators: {grid_search.best_params_['n_estimators']}")
print(f"Best MAE score: {-grid_search.best_score_:.2f}")

# Use the best model for predictions
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test_transformed)
mae = mean_absolute_error(y_test, y_pred)
print(f"Test MAE with best model: {mae:.2f}")

# Create the final pipeline with the best estimator for later use
final_pipeline = Pipeline(steps=[
    ('preprocessing', pipeline_without_model),
    ('model', best_rf)
])

Fitting 5 folds for each of 45 candidates, totalling 225 fits


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vec

Best n_estimators: 50
Best MAE score: 13.64
Test MAE with best model: 12.35


Exception ignored in: <function ResourceTracker.__del__ at 0x111169c60>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10e271c60>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 77, in __del__
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 86, in _stop
  File "/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/multiprocessing/resource_tracker.py", line 111, in _stop_locked
ChildProcessErr

In [77]:
print(f"Best parameters: {grid_search}")

Best parameters: GridSearchCV(cv=5, estimator=RandomForestRegressor(random_state=0), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'min_samples_leaf': [1, 3, 5],
                         'n_estimators': [50, 100, 200, 300, 500]},
             scoring='neg_mean_absolute_error', verbose=1)
